## GridMET Data Subsetting for Colorado

### What I’m doing here

In this step, I’m taking the GridMET climate data and narrowing it down so it only covers Colorado. The original files cover a much larger geographic area, but for this project I only need data within Colorado bounds.

---

### Bounding Box

I defined a bounding box that roughly matches Colorado:

- Longitude: -109.06 to -102.04  
- Latitude: 37.00 to 41.00  

This lets me filter out everything outside of Colorado.

---

### How the Subsetting Works

For each NetCDF file:

1. I load the dataset using `xarray`
2. I slice it using the latitude and longitude bounds
3. I identify the variable inside the file (like temperature or precipitation)
4. If it’s a temperature variable, I convert it from Kelvin to Celsius
5. I save the new subset file

---

### Important Note

GridMET stores latitude in descending order, so the slicing is done like this:

```python
lat=slice(lat_max, lat_min)

In [8]:
import glob
import xarray as xr

lon_min, lon_max = -109.06, -102.04
lat_min, lat_max =  37.00,  41.00

def subset_gridmet(nc_file):
    ds = xr.open_dataset(nc_file, engine="netcdf4")
    
    
    ds_co = ds.sel(
        lon=slice(lon_min, lon_max),
        lat=slice(lat_max, lat_min)
    )
    
    
    var_name = list(ds_co.data_vars)[0]
    print(f"{nc_file} → variable: {var_name}")
    
    
    if "temperature" in var_name:
        ds_co[f"{var_name}_c"] = ds_co[var_name] - 273.15
    
    # Save
    out_file = nc_file.replace(".nc", "_CO.nc")
    ds_co.to_netcdf(out_file)
    return out_file


gridmet_files = sorted(
    glob.glob("tmmx_*.nc") +
    glob.glob("tmmn_*.nc") +
    glob.glob("pr_*.nc")
)

outputs = [subset_gridmet(f) for f in gridmet_files]
print(f"\n Saved {len(outputs)} Colorado-only files")


pr_2018.nc → variable: precipitation_amount
pr_2019.nc → variable: precipitation_amount
pr_2020.nc → variable: precipitation_amount
pr_2021.nc → variable: precipitation_amount
pr_2022.nc → variable: precipitation_amount
pr_2023.nc → variable: precipitation_amount
pr_2024.nc → variable: precipitation_amount
tmmn_2018.nc → variable: air_temperature
tmmn_2019.nc → variable: air_temperature
tmmn_2020.nc → variable: air_temperature
tmmn_2021.nc → variable: air_temperature
tmmn_2022.nc → variable: air_temperature
tmmn_2023.nc → variable: air_temperature
tmmn_2024.nc → variable: air_temperature
tmmx_2018.nc → variable: air_temperature
tmmx_2019.nc → variable: air_temperature
tmmx_2020.nc → variable: air_temperature
tmmx_2021.nc → variable: air_temperature
tmmx_2022.nc → variable: air_temperature
tmmx_2023.nc → variable: air_temperature
tmmx_2024.nc → variable: air_temperature

 Saved 21 Colorado-only files
